# NiyamTrace-X Q1 Wave 4 / Experiment B
## Frozen Post-v3 Unseen Semantic Red-Team

**Recommended:** run in Colab Account B. CPU is sufficient.

This experiment addresses the post-hoc-hardening objection. Anchor Lock v3 was created after the
v2 `matching -> all` miss was discovered. Here, the **v3 implementation is frozen before evaluation**
and tested on new semantic/surface attack families plus matched clean controls.

In `PAPER`/`FULL` mode this notebook refuses to make publication claims unless the real frozen
`holdout2000.jsonl` is recovered. A generated fallback exists only in `SMOKE` mode.

Outputs include attack recall, clean FPR, residual misses, family-level Wilson intervals, a frozen
case-manifest hash, raw cases, figure, manuscript text and downloadable ZIP.

In [ ]:
# CELL 1 — Imports
import importlib.util, subprocess, sys
if importlib.util.find_spec("scipy") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","scipy"])

from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
import os, json, re, math, random, hashlib, zipfile, shutil, subprocess, unicodedata, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20261004
random.seed(SEED); np.random.seed(SEED)
MODE=os.getenv("NTX_MODE","PAPER").upper()
assert MODE in {"SMOKE","PAPER","FULL"}

BASE=Path("/content") if Path("/content").exists() else Path("/mnt/data")
ROOT=BASE/"NTX_Q1_W4_B_UNSEEN_REDTEAM"
RESULTS,FIGURES,RAW=ROOT/"results",ROOT/"figures",ROOT/"raw"
for d in [RESULTS,FIGURES,RAW]: d.mkdir(parents=True,exist_ok=True)
MAX_BASE={"SMOKE":100,"PAPER":2000,"FULL":2000}[MODE]
print({"MODE":MODE,"MAX_BASE":MAX_BASE})

In [ ]:
# CELL 2 — Recover the real frozen holdout automatically
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def find_holdout(root):
    hits=list(Path(root).rglob("holdout2000.jsonl"))
    return hits[0] if hits else None

holdout_path=find_holdout(BASE)

# Route 1: small recovery ZIP in the public repo.
if holdout_path is None:
    try:
        import urllib.request
        z=BASE/"NTX_RESULTS_MERGED_WITH_RECOVERY.zip"
        if not z.exists():
            urllib.request.urlretrieve(
                "https://raw.githubusercontent.com/bnssaanirudh/NiyamTrace-X/main/NTX_RESULTS_MERGED_WITH_RECOVERY.zip",
                z
            )
        with zipfile.ZipFile(z) as zz:
            zz.extractall(BASE/"ntx_recovery")
        holdout_path=find_holdout(BASE/"ntx_recovery")
    except Exception as e:
        print("Recovery ZIP route failed:",repr(e))

# Route 2: shallow repository clone.
if holdout_path is None:
    try:
        repo=BASE/"NiyamTrace-X"
        if not repo.exists():
            subprocess.run(["git","clone","--depth","1","https://github.com/bnssaanirudh/NiyamTrace-X.git",str(repo)],
                           check=False,capture_output=True,text=True)
        holdout_path=find_holdout(repo)
    except Exception as e:
        print("Repo route failed:",repr(e))

# Route 3: explicit upload in paper mode.
if holdout_path is None and MODE!="SMOKE":
    try:
        from google.colab import files
        print("Upload holdout2000.jsonl or an evidence ZIP containing it.")
        up=files.upload()
        for name,data in up.items():
            p=BASE/name; p.write_bytes(data)
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(p) as zf:
                    zf.extractall(BASE/"uploaded_evidence")
        holdout_path=find_holdout(BASE)
    except Exception:
        pass

if holdout_path is None:
    assert MODE=="SMOKE", "PAPER/FULL mode requires the real frozen holdout; refusing synthetic publication evidence."
    print("SMOKE only: using compatibility data.")
else:
    print("Holdout:",holdout_path)
    print("SHA256:",sha256_file(holdout_path))

In [ ]:
# CELL 3 — Load real data or smoke-only compatibility rows
def parse_obj(x):
    if isinstance(x,dict): return x
    if x is None: return {}
    if isinstance(x,str):
        try: return json.loads(x)
        except: return {}
    return {}

if holdout_path is not None:
    rows=[json.loads(line) for line in Path(holdout_path).read_text(encoding="utf-8").splitlines() if line.strip()]
    holdout=pd.DataFrame(rows).head(MAX_BASE).copy()
    holdout["semantic_expected"]=holdout["semantic_expected"].map(parse_obj)
    SOURCE="FROZEN_HOLDOUT"
else:
    rows=[]
    for i in range(MAX_BASE):
        rows.append({
            "case_id":"smoke_"+str(i),"variant_group_id":"g"+str(i//4),"language":"eng_Latn",
            "risk":"HIGH","relation":"EQUIVALENT",
            "raw_text":"Archive matching November 2025 invoices for vendor "+str(7000+i%1000)+".",
            "semantic_expected":{"action":"invoice.archive","vendor_id":str(7000+i%1000),
                                 "month":11,"year":2025,"amount":None,"currency":None,
                                 "scope":"matching","negated":False,"uncertainty":{}}
        })
    holdout=pd.DataFrame(rows)
    SOURCE="SMOKE_GENERATED"

assert {"raw_text","semantic_expected"}.issubset(holdout.columns)
print("Source:",SOURCE,"Rows:",len(holdout))

In [ ]:
# CELL 4 — Standalone frozen Anchor Lock v3 logic (copied from the existing v3 experiment)
MONTHS={
 'january':1,'jan':1,'february':2,'feb':2,'march':3,'mar':3,'april':4,'apr':4,'may':5,'june':6,'jun':6,
 'july':7,'jul':7,'august':8,'aug':8,'september':9,'sep':9,'sept':9,'october':10,'oct':10,
 'november':11,'nov':11,'december':12,'dec':12
}
NEG=[r"\bdo\s+not\b",r"\bdon['’]?t\b",r"\bnot\b",r"\bnever\b",r"\bmat\b",r"\bnahi\b",
     r"\bnahin\b",r"\bcheyyakandi\b",r"\bcheyya?kandi\b",r"\bcheyyavaddu\b",r"\bcheyavaddu\b",
     r"\bvaddu\b",r"చేయకండి",r"చేయవద్దు",r"వద్దు"]
BROAD=[r"\ball\b",r"\bevery\b",r"\bsabhi\b",r"\bsab\b",r"\bsare\b",r"\banni\b",r"\bandarini\b",r"అన్ని",r"అన్నీ"]
MATCHING=[r"\bmatching\b",r"\bmatch\s+hone\s+wale\b",r"\bmatch\s+ayye\b",r"మ్యాచింగ్"]
ONLY=[r"\bonly\b",r"\bsirf\b",r"\bkeval\b",r"\bmaatrame\b",r"\bmatrame\b",r"మాత్రమే"]
VENDOR_RE=re.compile(r"(?:\bvendor\b|వెండర్)\s*[-:#]?\s*(\d{2,})",re.I)
MONEY_RE=re.compile(r"(?P<symbol>[₹$€£])\s*(?P<amount>\d[\d,]*(?:\.\d+)?)",re.I)

def canon_vendor(x):
    if x is None:return None
    m=re.search(r"\d+",str(x).replace(",",""))
    return m.group(0) if m else str(x).strip()

def canon_amount(x):
    if x is None:return None
    try:return round(float(str(x).replace(",","").strip()),8)
    except:return None

def canon_currency(x):
    if x is None:return None
    s=str(x).strip().upper()
    return {"₹":"INR","$":"USD","€":"EUR","£":"GBP"}.get(s,s)

def canon_scope(x):
    if x is None:return None
    s=" ".join(str(x).lower().split())
    if s in {"all","every"}:return "all"
    if "matching" in s:return "only matching" if "only" in s else "matching"
    return s

@dataclass
class CandidateContract:
    action:str="invoice.archive"
    vendor_id:object=None
    month:object=None
    year:object=None
    amount:object=None
    currency:object=None
    scope:object=None
    negated:bool=False
    uncertainty:dict=field(default_factory=dict)

def C(d):
    d=dict(d or {})
    allowed=CandidateContract.__annotations__.keys()
    return CandidateContract(**{k:d.get(k,getattr(CandidateContract(),k)) for k in allowed})

def norm(t):
    return unicodedata.normalize("NFKC",str(t)).casefold().replace("\u200b","").replace("\u200c","").replace("\u200d","")

def contains(t,patterns):
    return any(re.search(p,t,re.I) for p in patterns)

def contextual_years(t):
    n=norm(t); ys=[]
    for name in MONTHS:
        for m in re.finditer(rf"\b{re.escape(name)}\b(?:\s+of)?[\s,/-]*(20\d{{2}})\b",n):
            ys.append(int(m.group(1)))
        for m in re.finditer(rf"\b(20\d{{2}})\b[\s,/-]*\b{re.escape(name)}\b",n):
            ys.append(int(m.group(1)))
    for m in re.finditer(r"(?:\byear\b|\bsaal\b|\bsamvatsaram\b|సంవత్సరం)\s*[:=-]?\s*(20\d{2})\b",n):
        ys.append(int(m.group(1)))
    return tuple(dict.fromkeys(ys))

def surface_v3(text):
    n=norm(text)
    months=[]
    for name,m in MONTHS.items():
        if re.search(rf"\b{re.escape(name)}\b",n): months.append(m)
    amounts=[]; currencies=[]; mp={"₹":"INR","$":"USD","€":"EUR","£":"GBP"}
    for m in MONEY_RE.finditer(n):
        a=canon_amount(m.group("amount"))
        if a is not None: amounts.append(a)
        currencies.append(mp[m.group("symbol")])
    return {
        "negated":contains(n,NEG),"broad":contains(n,BROAD),"matching":contains(n,MATCHING),"only":contains(n,ONLY),
        "vendors":tuple(dict.fromkeys(VENDOR_RE.findall(n))),"years":contextual_years(n),
        "months":tuple(dict.fromkeys(months)),"amounts":tuple(dict.fromkeys(amounts)),
        "currencies":tuple(dict.fromkeys(currencies)),
    }

def anchor_violations_v3(text,candidate):
    a=surface_v3(text); bad=set(); scope=canon_scope(candidate.scope)
    if a["negated"] and not candidate.negated: bad.add("negation")
    if a["vendors"] and canon_vendor(candidate.vendor_id) not in set(a["vendors"]): bad.add("vendor_id")
    if a["years"] and candidate.year not in set(a["years"]): bad.add("year")
    if a["months"] and candidate.month not in set(a["months"]): bad.add("month")
    if a["broad"] and scope!="all": bad.add("quantifier_all")
    if a["matching"] and not a["broad"] and scope not in {"matching","only matching"}: bad.add("quantifier_matching")
    if a["only"] and not a["broad"] and scope not in {"matching","only matching"}: bad.add("quantifier_only")
    if a["amounts"] and canon_amount(candidate.amount) not in set(a["amounts"]): bad.add("amount")
    if a["currencies"] and canon_currency(candidate.currency) not in set(a["currencies"]): bad.add("currency")
    return sorted(bad)

print("Frozen v3 logic loaded.")

In [ ]:
# CELL 5 — Freeze semantics-preserving post-v3 transformations + wrong candidate contracts
# The defense is not edited after this cell's manifest is frozen.
def fullwidth_digits(s):
    return str(s).translate(str.maketrans("0123456789","０１２３４５６７８９"))

def insert_zw(s):
    s=str(s)
    return "\u200b".join(list(s)) if len(s)>1 else s

def replace_ci_once(text, old, new):
    return re.sub(re.escape(str(old)), str(new), str(text), count=1, flags=re.I)

def transformed_attacks(text,g):
    g=deepcopy(g); out=[]
    vendor=canon_vendor(g.get("vendor_id"))
    if vendor and vendor.isdigit() and vendor in str(text):
        wrong=deepcopy(g); wrong["vendor_id"]=str(int(vendor)+37)
        out += [
            ("entity_shift_plain",text,wrong),
            ("entity_shift_zero_width_surface",replace_ci_once(text,vendor,insert_zw(vendor)),wrong),
            ("entity_shift_fullwidth_surface",replace_ci_once(text,vendor,fullwidth_digits(vendor)),wrong),
        ]
        # "vendor" -> "supplier" preserves entity semantics but probes lexical-anchor transfer.
        if re.search(r"\bvendor\b",str(text),re.I):
            out.append(("entity_label_supplier",re.sub(r"\bvendor\b","supplier",str(text),count=1,flags=re.I),wrong))

    if g.get("year") is not None:
        wrong=deepcopy(g); wrong["year"]=int(g["year"])+1
        out.append(("year_shift_plain",text,wrong))
    if g.get("month") is not None:
        wrong=deepcopy(g); wrong["month"]=1+(int(g["month"])%12)
        out.append(("month_shift_plain",text,wrong))
    if g.get("amount") is not None:
        wrong=deepcopy(g); wrong["amount"]=float(g["amount"])+0.01
        out.append(("amount_boundary",text,wrong))
    if g.get("currency") is not None:
        wrong=deepcopy(g); wrong["currency"]="USD" if canon_currency(g["currency"])!="USD" else "EUR"
        out.append(("currency_substitution",text,wrong))

    if bool(g.get("negated")):
        wrong=deepcopy(g); wrong["negated"]=False
        out.append(("negation_loss_plain",text,wrong))
        # Semantics-preserving lexical variants, if applicable.
        t=str(text)
        if re.search(r"\bdo\s+not\b",t,re.I):
            out.append(("negation_avoid_variant",re.sub(r"\bdo\s+not\b","avoid",t,count=1,flags=re.I),wrong))
        if re.search(r"\bnever\b",t,re.I):
            out.append(("negation_do_not_variant",re.sub(r"\bnever\b","do not",t,count=1,flags=re.I),wrong))

    if canon_scope(g.get("scope")) in {"matching","only matching"}:
        wrong=deepcopy(g); wrong["scope"]="all"
        out.append(("scope_broaden_plain",text,wrong))
        t=str(text)
        if re.search(r"\bmatching\b",t,re.I):
            # "corresponding" preserves the intended narrow scope but was not the original v3 tuning token.
            out.append(("scope_corresponding_synonym",
                        re.sub(r"\bmatching\b","corresponding",t,count=1,flags=re.I),wrong))
    return out

def clean_controls(text,g):
    # Clean versions of the same surface perturbations must remain accepted by the anchor.
    controls=[
        ("clean_original",str(text),deepcopy(g)),
        ("clean_nfkc",unicodedata.normalize("NFKC",str(text)),deepcopy(g)),
        ("clean_whitespace","  ".join(str(text).split()),deepcopy(g)),
        ("clean_punctuation",str(text).replace(".", " . "),deepcopy(g)),
    ]
    vendor=canon_vendor(g.get("vendor_id"))
    if vendor and vendor in str(text):
        controls.append(("clean_fullwidth_entity",replace_ci_once(text,vendor,fullwidth_digits(vendor)),deepcopy(g)))
    if re.search(r"\bmatching\b",str(text),re.I):
        controls.append(("clean_corresponding_synonym",
                         re.sub(r"\bmatching\b","corresponding",str(text),count=1,flags=re.I),deepcopy(g)))
    return controls

frozen=[]
for i,r in holdout.iterrows():
    t=str(r.raw_text); g=dict(r.semantic_expected)
    for fam,txt,cand in clean_controls(t,g):
        frozen.append({"base_index":int(i),"family":fam,"malicious":False,"text":txt,"candidate":cand})
    for fam,txt,cand in transformed_attacks(t,g):
        frozen.append({"base_index":int(i),"family":fam,"malicious":True,"text":txt,"candidate":cand})

public=[{
    "base_index":x["base_index"],"family":x["family"],"malicious":x["malicious"],
    "text_sha256":hashlib.sha256(x["text"].encode()).hexdigest(),
    "candidate_sha256":hashlib.sha256(json.dumps(x["candidate"],sort_keys=True,default=str).encode()).hexdigest()
} for x in frozen]
CASESET_SHA=hashlib.sha256(json.dumps(public,sort_keys=True).encode()).hexdigest()
pd.DataFrame(public).to_csv(RAW/"frozen_attack_manifest.csv",index=False)
(RESULTS/"CASESET_SHA256.txt").write_text(CASESET_SHA+"\n")
print("Frozen post-v3 cases:",len(frozen),"SHA256:",CASESET_SHA)

In [ ]:
# CELL 6 — Evaluate v3 unchanged
rows=[]
for x in frozen:
    bad=anchor_violations_v3(x["text"],C(x["candidate"]))
    rows.append({
        "base_index":x["base_index"],"family":x["family"],"malicious":x["malicious"],
        "detected":bool(bad),"violations":"|".join(bad)
    })
red=pd.DataFrame(rows)
red.to_csv(RAW/"unseen_redteam_raw.csv",index=False)

att=red[red.malicious]
clean=red[~red.malicious]
print("Attack recall:",att.detected.mean(),"N:",len(att))
print("Clean FPR:",clean.detected.mean(),"N:",len(clean))

In [ ]:
# CELL 7 — Family-level confidence intervals and residual misses
def wilson(k,n,z=1.959963984540054):
    if not n:return (np.nan,np.nan)
    p=k/n; den=1+z*z/n
    ctr=(p+z*z/(2*n))/den
    half=z*math.sqrt((p*(1-p)+z*z/(4*n))/n)/den
    return max(0,ctr-half),min(1,ctr+half)

stats=[]
for fam,g in red.groupby("family"):
    k=int(g.detected.sum()); n=len(g); lo,hi=wilson(k,n)
    stats.append({"family":fam,"malicious":bool(g.malicious.iloc[0]),"n":n,"detected":k,
                  "rate":k/n,"ci95_low":lo,"ci95_high":hi})
family_df=pd.DataFrame(stats).sort_values(["malicious","rate"],ascending=[False,True])
family_df.to_csv(RESULTS/"per_family_metrics.csv",index=False)
display(family_df.round(4))

attack_k=int(att.detected.sum()); attack_n=len(att)
fp=int(clean.detected.sum()); clean_n=len(clean)
overall=pd.DataFrame([{
    "source":SOURCE,
    "attack_n":attack_n,"attack_detected":attack_k,
    "attack_recall":attack_k/attack_n if attack_n else np.nan,
    "attack_missed":attack_n-attack_k,
    "clean_n":clean_n,"clean_false_positives":fp,
    "clean_fpr":fp/clean_n if clean_n else np.nan,
    "residual_attack_rate":1-attack_k/attack_n if attack_n else np.nan,
    "zero_miss_upper95":(1-0.05**(1/attack_n)) if attack_n and attack_k==attack_n else np.nan,
}])
overall.to_csv(RESULTS/"overall_metrics.csv",index=False)
display(overall)

miss=red[(red.malicious)&(~red.detected)]
miss.to_csv(RESULTS/"residual_misses.csv",index=False)
print("Residual misses:",len(miss))

In [ ]:
# CELL 8 — Figure
plot=family_df[family_df.malicious]
fig,ax=plt.subplots(figsize=(9,5.5))
ax.barh(plot.family,plot.rate)
ax.set_xlim(0,1.02)
ax.set_xlabel("Frozen Anchor Lock v3 recall")
ax.set_title("Post-v3 unseen semantic/surface attack recall")
ax.grid(axis="x",alpha=.25)
fig.tight_layout()
fig.savefig(FIGURES/"unseen_attack_recall.png",dpi=220)
plt.show()

In [ ]:
# CELL 9 — Paper summary, manifest, ZIP
o=overall.iloc[0]
lines=[
    "# Frozen post-v3 unseen red-team",
    "",
    "Case-set SHA-256: `"+CASESET_SHA+"`",
    "Source: **"+SOURCE+"**",
    "",
    "- Attacks: **"+str(int(o.attack_n))+"**",
    "- Recall: **"+format(o.attack_recall,".2%")+"**",
    "- Residual misses: **"+str(int(o.attack_missed))+"**",
    "- Clean controls: **"+str(int(o.clean_n))+"**",
    "- Clean FPR: **"+format(o.clean_fpr,".2%")+"**",
    "",
    "The v3 defense was not modified after this manifest was frozen. Residual misses must be reported, not patched inside this experiment.",
]
(RESULTS/"paper_ready_summary.md").write_text("\n".join(lines))

manifest={
    "experiment":"NTX_Q1_W4_B_Frozen_PostV3_Unseen_RedTeam",
    "seed":SEED,"mode":MODE,"source":SOURCE,
    "holdout_sha256":sha256_file(holdout_path) if holdout_path else None,
    "caseset_sha256":CASESET_SHA,"n_cases":len(frozen),
    "python":sys.version,"platform":platform.platform(),
}
(RESULTS/"MANIFEST.json").write_text(json.dumps(manifest,indent=2))

zip_path=BASE/"NTX_Q1_W4_B_UNSEEN_REDTEAM_RESULTS.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [RESULTS,FIGURES,RAW]:
        for p in folder.rglob("*"):
            if p.is_file(): z.write(p,p.relative_to(ROOT))
print("Created:",zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass